# Cross-validate the wound models on Kaggle

5-fold cross-validation (3-fold if 5 won't fit the time budget) of the **wound outline** (SegFormer-B2 at **768 px**) and
the **wound type** (ConvNeXt-Tiny, 384 px). Folds are grouped by patient and balanced (by wound size for the outline, by
wound type for the classifier). The same locked test set as the first run scores every fold, so the result is
**mean ± sd** instead of one split.

**Before you start:** Session options → Accelerator **GPU T4 x2** (two GPUs: two folds train at once), Internet **on**,
Persistence **Files only**. Then **Save Version → Save & Run All (Commit)** so it runs in the background.

The first step times a short trial on the GPU and decides between 5 and 3 folds (budget: 10 hours, Kaggle stops at 12).
If the run is cut short, run it again: finished folds are kept and the outline resumes per epoch.

At the end, download `outputs/`: `summary.md` has the table, `best/` holds the models to install.

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d wound_1 ] || git clone --depth 1 https://github.com/opalvedansh/wound_1.git
cd wound_1/wound-ai
pip install -q -r requirements.txt   # PyTorch is already installed on Kaggle
nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data

FUSeg (1,010 foot-ulcer photos with outlines) and AZH (930 photos labelled by wound type). Both are public
research datasets; check their terms before any commercial use. To add more outline data later (for example the
Mendeley lower-limb set as a private Kaggle dataset), add another `--pairs name:images_dir:masks_dir`.

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
mkdir -p data/raw && cd data/raw
if [ ! -d fuseg ]; then
  git clone -q --depth 1 --filter=blob:none --sparse https://github.com/uwm-bigdata/wound-segmentation.git src
  (cd src && git sparse-checkout set --no-cone "/data/Foot Ulcer Segmentation Challenge/train/" "/data/Foot Ulcer Segmentation Challenge/validation/")
  mv "src/data/Foot Ulcer Segmentation Challenge" fuseg && rm -rf src
fi
if [ ! -d azh ]; then
  mkdir azh && cd azh
  for f in Train Test; do
    curl -sL -o $f.zip https://raw.githubusercontent.com/uwm-bigdata/wound-classification-using-images-and-locations/main/dataset/$f.zip
    unzip -q $f.zip && rm $f.zip
  done
  rm -rf __MACOSX && cd ..
fi
ls fuseg/train/images | wc -l; ls azh/Train

In [ ]:
%%bash
set -e
cd /kaggle/working/wound_1/wound-ai
# Same seed and sources as on the Mac, so the splits match and test results are comparable.
python scripts/prepare_data.py \
  --pairs "fuseg:data/raw/fuseg/train/images:data/raw/fuseg/train/labels" \
          "fuseg:data/raw/fuseg/validation/images:data/raw/fuseg/validation/labels" \
  --out data/manifest.csv
python scripts/prepare_data.py \
  --classes "azh:data/raw/azh/Train:wound_type:configs/azh_map.json" "azh:data/raw/azh/Test:wound_type:configs/azh_map.json" \
  --out data/manifest_cls.csv

## 2. Cross-validation (several hours)

Prints the timing estimate and the number of folds first, then one line per epoch per fold (`[outline 2/5] epoch 14 …`).

In [ ]:
%%bash
cd /kaggle/working/wound_1/wound-ai
python scripts/cv.py --seg-manifest data/manifest.csv --cls-manifest data/manifest_cls.csv \
  --folds 5 --fallback-folds 3 --budget-hours 10 \
  --seg-size 768 --seg-batch 4 --seg-epochs 60 \
  --cls-size 384 --cls-batch 32 --cls-epochs 40 \
  --out /kaggle/working/cv

## 3. Package the results

In [ ]:
%%bash
set -e
mkdir -p /kaggle/working/outputs
cd /kaggle/working/cv
cp summary.md summary.json plan.json /kaggle/working/outputs/
cp -r best /kaggle/working/outputs/
for kind in seg cls; do
  for d in $kind/fold*; do
    f=$(basename $d)
    cp $d/eval_test.json /kaggle/working/outputs/${kind}_${f}_test.json 2>/dev/null || true
    cp $d/history.csv /kaggle/working/outputs/${kind}_${f}_history.csv 2>/dev/null || true
    cp $d/val_report.json /kaggle/working/outputs/${kind}_${f}_val.json 2>/dev/null || true
  done
done
cat /kaggle/working/outputs/summary.md
ls -la /kaggle/working/outputs /kaggle/working/outputs/best